In [6]:
import os
import yaml
import neo4j
from dotenv import load_dotenv


load_dotenv(r"../../backend/.env")

True

In [7]:
yml_path = r"C:\Users\karthik.kishor\Desktop\Projects\mti_brain\brain\semantic_model_generator\output\lpp_semantic_model.yml"

In [8]:
NEO4J_URI=os.getenv("NEO4J_URI")
NEO4J_USER=os.getenv("NEO4J_USER")
NEO4J_PASSWORD=os.getenv("NEO4J_PASSWORD")
NEO4J_DB=os.getenv("NEO4J_DB")

Node details


Column
Key
Value
<id>
4:04d52f91-396f-4999-8da3-b9069171d431:1900

cohere_embedding
[-0.00433533, -0.02442864, 0.046444576, -0.0101785995, -0.05579381, -0.00022030072, -0.056698572, 0.03030961, 0.008218277, 0.027293729, -0.021563552, … Show all

created_at
"2026-06-02T04:25:58.493642+00:00"

data_type
"character varying"

default_aggregation
"NONE"

description
"Reference code identifying the internal operating account into which settlement funds from this acquirer are deposited."

distinct_values
["USA_RGNL_OPERATING", "EUR_RGNL_OPERATING", "GR_US_INC_OP_4"]

embedding_generated_at
"2026-06-02T06:29:12.647991+00:00"

embedding_model
"arn:aws:bedrock:us-west-2:295790629637:inference-profile/global.cohere.embed-v4:0"

enrichment_status
"complete"

filter_selectivity
"low"

has_data
true

id
"lpp.acquirer.settlement_account_ref"

is_foreign_key
false

is_groupable
true

is_measurable
false

is_nullable
true

is_pii
false

is_pk
false

is_surrogate_fk
false

is_surrogate_key
false

n_distinct
-0.375

name
"settlement_account_ref"

null_frac
0.0

ordinal_position
5

pii_type
""

referenced_column
"code"

referenced_table_fqn
"lpp.bank_account"

same_name_col_count
2

sample_values
["USA_RGNL_OPERATING", "EUR_RGNL_OPERATING", "GR_US_INC_OP_4"]

semantic_type
"code"

source_hash
"a9d63b41360b217cf1050649fab33d16"

synonyms
["settlement account", "operating account", "payout account reference"]

table_fqn
"lpp.acquirer"

temporal_grain
"none"

top_freq_values
[]

updated_at
"2026-06-02T06:29:12.647991+00:00"

value_aliases
["USA_RGNL_OPERATING -> US Regional Operating Account", "EUR_RGNL_OPERATING -> EUR Regional Operating Account", "GR_US_INC_OP_4 -> US Inc Operating Ac… Show all

value_vocabulary
["USA_RGNL_OPERATING", "EUR_RGNL_OPERATING", "GR_US_INC_OP_4"]

version
1



Node details


Table
Key
Value
<id>
4:04d52f91-396f-4999-8da3-b9069171d431:1793

betweenness_score
167.57679738682174

business_domain
"banking"

cohere_embedding
[-0.029382443, -0.021456016, 0.053298384, -0.040452108, -0.040178783, -0.03580558, -0.036625557, 0.009908033, 0.00076018524, -0.0021438934, -0.0259658… Show all

column_count
14

community_id
9

created_at
"2026-06-02T04:25:57.550176+00:00"

description
"Represents individual bank fee charges levied by banks against specific accounts for banking services during a statement period, including the actual… Show all

embedding_generated_at
"2026-06-02T06:29:28.673862+00:00"

embedding_model
"arn:aws:bedrock:us-west-2:295790629637:inference-profile/global.cohere.embed-v4:0"

enrichment_status
"complete"

fqn
"lpp.bank_fee"

grain
"One row per bank fee charge for a specific service on a bank account within a statement period."

has_seasonality_pattern
false

hub_join_col
""

in_degree
16.0

intent_tags
["counterparty_exposure", "general_analytics", "multi_entity_join", "scenario_forecast", "trend_analysis"]

is_dimension_hub
false

is_rollup
false

is_subquery_anchor
true

is_time_series
true

is_view
false

name
"bank_fee"

natural_dimensions
["uuid", "bank_ref", "bank_account_ref", "service_code", "statement_period", "charge_date", "currency_code", "cash_flow_ref", "rate_card_ref", "flagge… Show all

natural_measures
["units", "charged_amount", "expected_amount", "overage_amount"]

ontology_class
"lpp:BankFee"

out_degree
24.0

pagerank_score
1.3612042292605586

pk_columns
["uuid"]

row_count
12597

scc_id
0

schema
"lpp"

sortkey1
"charge_date"

source_hash
"13a237c66a19bd76c344c0f830152a64"

synonyms
["Bank Fee", "Bank Charge", "Account Service Fee", "Banking Cost", "Bank Statement Fee"]

table_type
"fact"

time_dimension_col
"charge_date"

time_dimension_grain
"day"

triangle_count
470

typical_join_role
"anchor"

typical_lookback_days
0

updated_at
"2026-06-02T06:29:28.673862+00:00"

version
1

wcc_component_id
0


MATCH(t:Table)-[:HAS_COLUMN]->(c:Column) RETURN t, c LIMIT 10

In [9]:

# Connect to Neo4j and fetch table + column descriptions
driver = neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session(database=NEO4J_DB) as session:
    # Fetch table descriptions: fqn = "schema.table"
    table_result = session.run(
        "MATCH (t:Table) WHERE t.fqn IS NOT NULL AND t.description IS NOT NULL "
        "RETURN t.fqn AS fqn, t.description AS description"
    )
    table_descriptions = {row["fqn"]: row["description"] for row in table_result}

    # Fetch column descriptions: id = "schema.table.column"
    col_result = session.run(
        "MATCH (c:Column) WHERE c.id IS NOT NULL AND c.description IS NOT NULL "
        "RETURN c.id AS id, c.description AS description"
    )
    col_descriptions = {row["id"]: row["description"] for row in col_result}

driver.close()

print(f"Fetched {len(table_descriptions)} table descriptions")
print(f"Fetched {len(col_descriptions)} column descriptions")


Fetched 105 table descriptions
Fetched 1110 column descriptions


In [11]:

import copy
import os

# Load the existing semantic model YAML
with open(yml_path, "r", encoding="utf-8") as f:
    model = yaml.safe_load(f)

# Enrich tables and columns with Neo4j descriptions
for table in model.get("tables", []):
    schema = table.get("schema", "lpp")
    name = table.get("name", "")
    fqn = f"{schema}.{name}"

    if fqn in table_descriptions:
        table["description"] = table_descriptions[fqn]

    for col in table.get("columns", []):
        col_name = col.get("name", "")
        col_id = f"{fqn}.{col_name}"
        if col_id in col_descriptions:
            col["description"] = col_descriptions[col_id]

# Write enriched YAML with UTF-8 encoding to handle special characters
out_path = os.path.join(os.path.dirname(yml_path), "lpp_semantic_model_with_descriptions.yml")
with open(out_path, "w", encoding="utf-8") as f:
    yaml.dump(model, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

print(f"Written to: {out_path}")


Written to: C:\Users\karthik.kishor\Desktop\Projects\mti_brain\brain\semantic_model_generator\output\lpp_semantic_model_with_descriptions.yml
